# Supplementary Table 4

Enterotype Dysbiosis Score (EDS) displays in the PREDICT cohort: the earlier 20-feature, 259-observation selection with its 85-patient display, and the three post hoc metabolite-enzyme pairs of Supplementary Fig. 10a with their associations, saved full-family q values and 192 descriptive adjustment estimates. This notebook rebuilds all eleven sheets from `source_data/`, compares them cell for cell with the published workbook (clinical columns withheld), and recomputes the statistics the table reports.

Run from the repository root: `python paper.py build STab4`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
d = panels.load(REPO)
ctrl = panels.load_controlled(d)

## Table

Eleven sheets. All20_features_259_observations to 85_patient_z: the earlier selection of 20 features (host proteins, microbial KEGG pathways, metabolites) across 259 matched observations (repeated visits included) and its display in 85 patients, one observation per patient (Fig. 4h, Supplementary Fig. 10b). Focused_raw to Focused_adjustments: the three metabolite-enzyme pairs of Supplementary Fig. 10a. Supports the legend (v28d line 481) and "examined three metabolite–enzyme pairs selected post hoc from an exploratory screen of this cohort (Supplementary Fig. 10a, Supplementary Table 4)" (v28d line 137). Unit: patient (85) or observation (259). No test in this cell.

In the repository version, acquisition and microbiome sample identifiers, clinical group and antibiotic state are withheld; sample identifiers are replaced by observation index (1-259) or patient display position (1-85, EDS order).

In [ ]:
sheets = panels.build_sheets(d)
path = panels.write_workbook(sheets, OUT / "Supplementary_Table_4.xlsx")
for name, rows in sheets.items():
    print(f"{name:34s} {len(rows):4d} rows x {max(len(r) for r in rows):4d} columns")

## Statistics: the three pairs

Spearman correlation of each metabolite with the protein signal of its enzyme (KO), over patients with positive finite values for both (n = 85, 84, 85 patients; two-sided t test, df = n - 2). "Spearman ρ=−0.53, −0.44 and −0.35, with 84 patients for tryptophan" (v28d line 137). q values are recomputed by Benjamini-Hochberg over the complete native KO linked-pair family (1,042 tested pairs) from the saved screen P values; the joint-adjusted model is the same family under adjustment for human-assigned share, protein-group depth, antibiotic status and clinical group (v28d line 137: "none passed multiple-testing correction after joint adjustment").

In [ ]:
una = panels.stat_focused_unadjusted(d)
print(una.to_string(index=False))
fq, qdiff = panels.stat_family_q(d)
print(fq.to_string(index=False))
print("largest |recomputed q - saved q_augmented| over the whole screen:", qdiff)
sheet_pairs = d["selected_pair_summary"]
assert np.allclose(una.rho, sheet_pairs.rho, rtol=0, atol=1e-12) and (una.n.to_numpy() == sheet_pairs.n.to_numpy()).all()
assert np.allclose(fq[fq.model == "unadjusted"].q, sheet_pairs.full_family_q, rtol=0, atol=1e-12)
assert np.allclose(fq[fq.model != "unadjusted"].q, sheet_pairs.joint_adjusted_full_family_q, rtol=0, atol=1e-12)
assert np.allclose(fq[fq.model != "unadjusted"].rho, sheet_pairs.joint_adjusted_rho, rtol=0, atol=1e-12)
joint = panels.stat_joint_adjusted(d, ctrl)
if joint is not None:
    print(joint.to_string(index=False))
    assert np.allclose(joint.rho, sheet_pairs.joint_adjusted_rho, rtol=0, atol=1e-12)

## Statistics: the six displayed features

Spearman correlation of each displayed feature with EDS over patients with a positive value (n = 85 or 84 patients), and Benjamini-Hochberg q recomputed in the complete native unadjusted EDS family of its layer (283 metabolites, 6,965 KOs). Sheet Focused_features (v28d line 481: "the feature associations").

In [ ]:
ff = panels.stat_focused_features(d)
print(ff.to_string(index=False))
assert np.allclose(ff.rho, d["focused_features"].EDS_rho, rtol=0, atol=1e-12)
assert np.allclose(ff.q, d["focused_features"].EDS_full_family_q, rtol=0, atol=1e-12)
assert (ff.n.to_numpy() == d["focused_features"].n_positive.to_numpy()).all()

## Statistics: 192 adjustment estimates

Partial rank correlations for all 16 combinations of human-assigned share, log2 protein-group depth, antibiotic status and clinical group, under native and annotated-relative normalization, on the pair-specific complete subsets (85, 84, 85 patients) and the common 84-patient subset: 3 x 2 x 2 x 16 = 192 point estimates, without P values or intervals (v28d line 799). "Adjustment for protein-group depth alone reduced them to ρ=−0.24, −0.06 and −0.10" (v28d line 137). Without the controlled input only the 48 estimates that use neither antibiotic status nor clinical group are recomputed.

In [ ]:
grid, n_common = panels.stat_adjustment_grid(d, ctrl)
ref = d["separate_adjustments_all_models"]
assert (grid[["metabolite", "KO", "scope", "normalization", "model", "n"]].to_numpy() == ref[["metabolite", "KO", "scope", "normalization", "model", "n"]].to_numpy()).all()
ok = np.isfinite(grid.rho.to_numpy())
print("estimates recomputed:", int(ok.sum()), "of", len(grid), "; common subset:", n_common)
print("largest |recomputed - sheet| rho:", float(np.max(np.abs(grid.rho.to_numpy()[ok] - ref.rho.to_numpy()[ok]))))
depth = grid[(grid.scope == "pair_complete") & (grid.normalization == "native") & (grid.model.isin(["none", "depth"]))]
print(depth.to_string(index=False))

## Statistics: overview features and z scores

Fixed_feature_key: positive patients and Spearman rho with EDS over the 85 patients (19 of 20 features have a positive value; HV366 has none). 85_patient_z and Focused_raw display values: within-feature z over positive finite values, population SD (v28d line 801).

In [ ]:
ov = panels.stat_overview_features(d)
print(ov.to_string(index=False))
assert (ov.n_positive == ov.n_sheet).all() and np.nanmax(ov.abs_diff) < 1e-12
print(panels.stat_z(d))

## Statistics: five-pair decomposition (Source Data only)

v28d line 799 describes an earlier decomposition of five metabolite-KO or metabolite-pathway pairs (all 16 covariate combinations, a common 84-patient subset, 1,000 patient bootstrap resamples). It is not part of this workbook; its saved model table is read here to trace the counts. The bootstrap is not rerun (see OPEN_ITEMS.md).

In [ ]:
five = panels.stat_five_pair(d)
print(five)

## Checks and numbers

Cell-for-cell comparison of the rebuilt workbook with the published workbook (redacted copy in `expected/`). With `FASTALAKE_CONTROLLED_DIR`, the complete published layout is rebuilt and compared with the published workbook itself. Then every number of `NUMBERS.tsv`, in row order.

In [ ]:
(UNIT / "checks").mkdir(exist_ok=True)
cmp = panels.compare_workbooks(OUT / "Supplementary_Table_4.xlsx", UNIT / "expected" / "Supplementary_Table_4_redacted.xlsx")
print(cmp.to_string(index=False))
cmp.to_csv(UNIT / "checks" / "workbook_vs_published_redacted.tsv", sep="\t", index=False)
assert cmp.differing.sum() == 0 and (cmp.rows == cmp.reference_rows).all()
if ctrl is not None:
    full = panels.write_workbook(panels.build_sheets(d, ctrl), OUT / "Supplementary_Table_4_full.xlsx")
    cmpf = panels.compare_workbooks(full, ctrl["published"])
    print(cmpf.to_string(index=False))
    cmpf.to_csv(UNIT / "checks" / "workbook_vs_published_full.tsv", sep="\t", index=False)
    assert cmpf.differing.sum() == 0 and (cmpf.rows == cmpf.reference_rows).all()
else:
    print("full-layout check SKIPPED: controlled input absent (committed receipt kept)")

cnt = panels.stat_counts(d, sheets)
dep = depth[depth.model == "depth"].set_index("metabolite").rho
grid_models = grid.groupby(["metabolite", "scope", "normalization"]).model.nunique().unique()
assert len(grid_models) == 1
pair_n = una.set_index("metabolite").n
nums = [
    panels.word(cnt["focused_pairs"]),                       # 137 three
    cnt["patients"],                                          # 137 85
    *una.rho.tolist(),                                        # 137 rho x3
    int(pair_n["HILIC_FT24070"]),                             # 137 84
    *[float(dep[m]) for m, *_ in panels.PAIRS],                      # 137 depth-only rho x3
    cnt["original_features"], cnt["original_observations"], cnt["patients"],   # 481
    panels.word(cnt["focused_pairs"]), panels.word(cnt["focused_measurements"]), cnt["adjustment_estimates"],
    panels.word(five["pairs"]), five["combinations"], five["common_n"], five["draws"],   # 799 five-pair
    panels.word(cnt["focused_pairs"]), int(grid_models[0]),   # 799 three pairs, 16 combinations
    *[int(pair_n[m]) for m, *_ in panels.PAIRS], n_common, len(grid),   # 799 85, 84, 85, 84, 192
]
for i, v in enumerate(nums, 1):
    print(f"{i}\t{v}")